# Make videos from frames
We will use the PNG biplots and cos_plots images to create videos. **Note** These images MUST be PNG format 

In [9]:
import re  # Ensure 're' is imported
import os
from pathlib import Path
import cv2
import numpy as np

def _extract_timestamp_from_filename(filename):
    """
    Extract float timestamp from filenames like:
    '<anyprefix>_event_<event>_<timestamp>_<n_vars>.png'.
    Examples: 'biplot_event_one_20.84_7_vars.png', 'cos2_event_one_20.84_7_vars.png'
    Returns a float for sorting. If not matched, returns +inf to push it to the end.
    """
    # Match regardless of prefix; capture the number positioned after the event name
    match = re.search(r"_event_.*?_(\d+(?:\.\d+)?)_\d+_vars\.png$", filename)
    if match:
        try:
            return float(match.group(1))
        except ValueError:
            pass
    return float("inf")

def make_video_and_store(
    image_folder,
    video_name,
    fps,
    max_width=3840,
    max_height=2160,
    codec_preference=None,
    allow_upscale=False,
    slow_start=None,
    slow_end=None,
    slow_factor=4,
):
    """
    Read PNG frames from image_folder, sort by timestamp in filename,
    resize to fit within max_width x max_height (keeping aspect), ensure even
    dimensions for codec compatibility, and save an MP4 under
    '<parent_of_image_folder>/videos/<video_name>'.

    - Uses high-quality interpolation: AREA for downscaling, LANCZOS4 for upscaling.
    - Tries a list of codecs (H.264 variants) with fallback to 'mp4v'.
    - If slow_start and slow_end are provided, images with timestamps in [slow_start, slow_end]
      are repeated slow_factor times to create a slow-motion effect.
    """
    image_folder_path = Path(image_folder)
    if not image_folder_path.exists():
        raise FileNotFoundError(f"Image folder not found: {image_folder}")

    images = [f for f in os.listdir(image_folder) if f.lower().endswith(".png")]
    if not images:
        raise FileNotFoundError(f"No PNG images found in {image_folder}")

    # Sort images by extracted timestamp (prefix-agnostic)
    images.sort(key=_extract_timestamp_from_filename)

    # Prepare list of (filename, timestamp) for slowmotion logic
    image_tuples = []
    for fname in images:
        ts = _extract_timestamp_from_filename(fname)
        image_tuples.append((fname, ts))

    # Prepare the output frame sequence, repeating frames in the slowmotion interval
    output_sequence = []
    for fname, ts in image_tuples:
        if (
            slow_start is not None
            and slow_end is not None
            and slow_start <= ts <= slow_end
        ):
            output_sequence.extend([fname] * slow_factor)
        else:
            output_sequence.append(fname)

    # Read the first frame
    first_frame = cv2.imread(str(image_folder_path / output_sequence[0]))
    if first_frame is None:
        raise RuntimeError(f"Failed to read first image: {output_sequence[0]}")
    height, width, layers = first_frame.shape

    # Compute target size within constraints and ensure even dimensions
    scale_limit = min(max_width / width, max_height / height)
    scale = min(scale_limit, 1.0) if not allow_upscale else scale_limit
    target_width = int(round(width * scale))
    target_height = int(round(height * scale))
    # enforce even values
    if target_width % 2 != 0:
        target_width -= 1
    if target_height % 2 != 0:
        target_height -= 1
    if target_width <= 0 or target_height <= 0:
        raise ValueError("Computed target video size is invalid. Check max_width/max_height.")

    if (target_width, target_height) != (width, height):
        interp_first = cv2.INTER_AREA if (target_width < width or target_height < height) else cv2.INTER_LANCZOS4
        first_frame = cv2.resize(first_frame, (target_width, target_height), interpolation=interp_first)

    # Create '<parent_of_image_folder>/videos' directory
    parent_dir = image_folder_path.parent
    videos_dir = parent_dir / 'videos'
    videos_dir.mkdir(parents=True, exist_ok=True)

    video_path = videos_dir / video_name

    # Try codec preferences in order
    if codec_preference is None:
        codec_preference = ['avc1', 'H264', 'X264', 'mp4v']

    writer = None
    chosen_codec = None
    for codec in codec_preference:
        tmp_writer = cv2.VideoWriter(str(video_path), cv2.VideoWriter_fourcc(*codec), fps, (target_width, target_height))
        if tmp_writer.isOpened():
            writer = tmp_writer
            chosen_codec = codec
            break
        else:
            tmp_writer.release()
    if writer is None:
        raise RuntimeError("Failed to initialize VideoWriter with provided codecs: " + ", ".join(codec_preference))

    try:
        # Write all frames in output_sequence (first already loaded)
        for idx, image_name in enumerate(output_sequence):
            if idx == 0:
                frame = first_frame
            else:
                frame = cv2.imread(str(image_folder_path / image_name))
                if frame is None:
                    continue
                if frame.shape[1] != target_width or frame.shape[0] != target_height:
                    interp = cv2.INTER_AREA if (frame.shape[1] > target_width or frame.shape[0] > target_height) else cv2.INTER_LANCZOS4
                    frame = cv2.resize(frame, (target_width, target_height), interpolation=interp)
            writer.write(frame)
    finally:
        writer.release()
        cv2.destroyAllWindows()

    return str(video_path)


def make_side_by_side_video_and_store(
    left_folder,
    right_folder,
    video_name,
    fps,
    max_width=3840,
    max_height=2160,
    codec_preference=None,
    allow_upscale=False,
    slow_start=None,
    slow_end=None,
    slow_factor=4,
    gap_px=10,
    bg_color=(0, 0, 0),
):
    """
    Create a side-by-side video combining frames from left_folder and right_folder
    matched by timestamp. Left frames (e.g., biplots) appear on the left, right frames
    (e.g., cos_plots) on the right with an optional gap between them. The resulting
    video is written to '<common_results_dir>/videos/<video_name>'.

    - Timestamps are extracted with _extract_timestamp_from_filename and matched 1:1
      (intersection). Order is ascending by timestamp.
    - Slow motion: If slow_start and slow_end are provided, any matching timestamp in
      [slow_start, slow_end] will repeat the pair slow_factor times.
    - Sizing: Both panes are resized to the same target height so their sum of widths
      plus gap fits within max_width. Height also respects max_height and, unless
      allow_upscale=True, will not exceed each original pane's height.
    - Even dimensions are enforced for codec compatibility.
    """
    left_path = Path(left_folder)
    right_path = Path(right_folder)
    if not left_path.exists() or not left_path.is_dir():
        raise FileNotFoundError(f"Left folder not found or not a directory: {left_folder}")
    if not right_path.exists() or not right_path.is_dir():
        raise FileNotFoundError(f"Right folder not found or not a directory: {right_folder}")

    left_images = [f for f in os.listdir(left_path) if f.lower().endswith('.png')]
    right_images = [f for f in os.listdir(right_path) if f.lower().endswith('.png')]
    if not left_images:
        raise FileNotFoundError(f"No PNG images found in left folder: {left_folder}")
    if not right_images:
        raise FileNotFoundError(f"No PNG images found in right folder: {right_folder}")

    # Map timestamps to filenames
    left_map = {}
    for f in left_images:
        ts = _extract_timestamp_from_filename(f)
        left_map[ts] = f
    right_map = {}
    for f in right_images:
        ts = _extract_timestamp_from_filename(f)
        right_map[ts] = f

    # Intersect timestamps and sort
    matched_ts = sorted(set(left_map.keys()).intersection(right_map.keys()))
    if not matched_ts:
        raise RuntimeError("No matching timestamps found between left and right folders.")

    # Build output sequence of timestamps with slow-motion repetition
    ts_sequence = []
    for ts in matched_ts:
        if (
            slow_start is not None
            and slow_end is not None
            and slow_start <= ts <= slow_end
        ):
            ts_sequence.extend([ts] * slow_factor)
        else:
            ts_sequence.append(ts)

    # Determine target pane sizes using first matched pair
    first_left = cv2.imread(str(left_path / left_map[matched_ts[0]]))
    first_right = cv2.imread(str(right_path / right_map[matched_ts[0]]))
    if first_left is None or first_right is None:
        raise RuntimeError("Failed to read the first matching image pair.")

    Hl, Wl = first_left.shape[0], first_left.shape[1]
    Hr, Wr = first_right.shape[0], first_right.shape[1]

    ratio_l = Wl / Hl
    ratio_r = Wr / Hr

    # Height constraints
    h_cap_left = float('inf') if allow_upscale else Hl
    h_cap_right = float('inf') if allow_upscale else Hr

    # Width-derived height limit to fit side-by-side
    if (ratio_l + ratio_r) <= 0:
        raise RuntimeError("Invalid aspect ratios for left/right frames.")
    height_limit_by_width = (max_width - gap_px) / (ratio_l + ratio_r)

    target_height = int(round(min(h_cap_left, h_cap_right, max_height, height_limit_by_width)))
    if target_height <= 0:
        raise ValueError("Computed target height is invalid. Check max_width/max_height.")
    if target_height % 2 != 0:
        target_height -= 1

    # Adjust height downward if combined width exceeds max_width or widths become invalid
    def compute_scaled_widths(th):
        wl = int(round(ratio_l * th))
        wr = int(round(ratio_r * th))
        if wl % 2 != 0:
            wl -= 1
        if wr % 2 != 0:
            wr -= 1
        return max(wl, 2), max(wr, 2)

    width_l, width_r = compute_scaled_widths(target_height)
    while (width_l + gap_px + width_r) > max_width:
        target_height -= 2
        if target_height <= 0:
            raise ValueError("Unable to fit side-by-side frames within max_width.")
        width_l, width_r = compute_scaled_widths(target_height)

    combined_width = width_l + gap_px + width_r
    if combined_width % 2 != 0:
        # Make combined width even by reducing right pane by 2 if possible
        if width_r > 2:
            width_r -= 2
        else:
            width_l -= 2
        combined_width = width_l + gap_px + width_r

    # Determine videos directory under shared 'results'
    left_results_dir = left_path.parents[1] if len(left_path.parents) >= 2 else left_path.parent
    right_results_dir = right_path.parents[1] if len(right_path.parents) >= 2 else right_path.parent
    if left_results_dir != right_results_dir:
        common_dir = Path(os.path.commonpath([str(left_path), str(right_path)]))
        videos_dir = common_dir / 'videos'
    else:
        videos_dir = left_results_dir / 'videos'
    videos_dir.mkdir(parents=True, exist_ok=True)

    # Initialize writer
    if codec_preference is None:
        codec_preference = ['avc1', 'H264', 'X264', 'mp4v']

    video_path = videos_dir / video_name
    writer = None
    for codec in codec_preference:
        tmp = cv2.VideoWriter(str(video_path), cv2.VideoWriter_fourcc(*codec), fps, (combined_width, target_height))
        if tmp.isOpened():
            writer = tmp
            break
        else:
            tmp.release()
    if writer is None:
        raise RuntimeError("Failed to initialize VideoWriter with provided codecs: " + ", ".join(codec_preference))

    try:
        for ts in ts_sequence:
            left_img = cv2.imread(str(left_path / left_map[ts]))
            right_img = cv2.imread(str(right_path / right_map[ts]))
            if left_img is None or right_img is None:
                continue

            # Resize to target pane sizes
            interp_l = cv2.INTER_AREA if (left_img.shape[1] > width_l or left_img.shape[0] > target_height) else cv2.INTER_LANCZOS4
            interp_r = cv2.INTER_AREA if (right_img.shape[1] > width_r or right_img.shape[0] > target_height) else cv2.INTER_LANCZOS4
            left_resized = cv2.resize(left_img, (width_l, target_height), interpolation=interp_l)
            right_resized = cv2.resize(right_img, (width_r, target_height), interpolation=interp_r)

            # Compose final frame with gap
            frame = np.full((target_height, combined_width, 3), bg_color, dtype=np.uint8)
            frame[:, 0:width_l] = left_resized
            frame[:, width_l + gap_px: width_l + gap_px + width_r] = right_resized

            writer.write(frame)
    finally:
        writer.release()
        cv2.destroyAllWindows()

    return str(video_path)

In [7]:
# Configure input/output paths and parameters
image_folder = (Path.cwd() / 'results' / 'biplots' / 'event_one')
fps = 10.0

# Build video file name with fps included
fps_str = str(int(fps)) if float(fps).is_integer() else str(fps)
video_name = f"biplots_eye_head_hand_car_event_1_{fps_str}fps.mp4"

# Optional slow-motion settings (adjust as needed)
slow_start = 23.0
slow_end = 24.0
slow_factor = 4

# Sanity checks and info
if not image_folder.exists():
    raise FileNotFoundError(f"Image folder not found: {image_folder}")
if not image_folder.is_dir():
    raise NotADirectoryError(f"Image folder is not a directory: {image_folder}")

# Check for legacy sibling 'video' directory; we will still write to 'videos'
legacy_video_dir = image_folder.parent / 'video'
if legacy_video_dir.exists() and legacy_video_dir.is_dir():
    print(f"Warning: sibling legacy folder exists: {legacy_video_dir}. Output will be written to {image_folder.parent / 'videos'}.")

images_count = sum(1 for f in os.listdir(image_folder) if f.lower().endswith('.png'))
print(f"Input folder: {image_folder}")
print(f"Images found: {images_count}")
print(f"Output folder (sibling): {image_folder.parent / 'videos'}")


Input folder: /Users/johnmadrid/GitHub/pca-driving-behavior/results/biplots/event_one
Images found: 508
Output folder (sibling): /Users/johnmadrid/GitHub/pca-driving-behavior/results/biplots/videos


In [8]:
# Create video
output_path = make_video_and_store(
    image_folder=image_folder,
    video_name=video_name,
    fps=fps,
    slow_start=slow_start,
    slow_end=slow_end,
    slow_factor=slow_factor,
)
print(f"Saved video to: {output_path}")

Saved video to: /Users/johnmadrid/GitHub/pca-driving-behavior/results/biplots/videos/biplots_eye_head_hand_car_event_1_10fps.mp4


In [10]:
# Side-by-side biplots | cos_plots video configuration and generation
left_folder = (Path.cwd() / 'results' / 'biplots' / 'event_one')
right_folder = (Path.cwd() / 'results' / 'cos_plots' / 'event_one')

# Use same fps and slow-motion settings as above
fps_str = str(int(fps)) if float(fps).is_integer() else str(fps)
video_name_sbs = f"biplots_cosplots_event_1_{fps_str}fps.mp4"

gap_px = 10  # small gap between the panes

# Info and optional legacy 'video' dir warning at common results level
results_dir_common = left_folder.parents[1]
legacy_results_video_dir = results_dir_common / 'video'
if legacy_results_video_dir.exists() and legacy_results_video_dir.is_dir():
    print(f"Warning: legacy folder exists: {legacy_results_video_dir}. Output will be written to {results_dir_common / 'videos'}.")

print(f"Left folder: {left_folder}")
print(f"Right folder: {right_folder}")
print(f"Output folder: {results_dir_common / 'videos'}")
print(f"Video filename: {video_name_sbs}")

# Create side-by-side video
output_path_sbs = make_side_by_side_video_and_store(
    left_folder=left_folder,
    right_folder=right_folder,
    video_name=video_name_sbs,
    fps=fps,
    slow_start=slow_start,
    slow_end=slow_end,
    slow_factor=slow_factor,
    gap_px=gap_px,
)
print(f"Saved side-by-side video to: {output_path_sbs}")


Left folder: /Users/johnmadrid/GitHub/pca-driving-behavior/results/biplots/event_one
Right folder: /Users/johnmadrid/GitHub/pca-driving-behavior/results/cos_plots/event_one
Output folder: /Users/johnmadrid/GitHub/pca-driving-behavior/results/videos
Video filename: biplots_cosplots_event_1_10fps.mp4
Saved side-by-side video to: /Users/johnmadrid/GitHub/pca-driving-behavior/results/videos/biplots_cosplots_event_1_10fps.mp4


In [30]:
# Video frame extraction & three-panel composite config
from pathlib import Path

# Source video (absolute path)
source_video = Path('/Users/johnmadrid/Documents/PhD/EEDA_experiment copy/Events_Videos/event1-new.mp4')

# Where to write extracted frames
video_frames_dir = Path.cwd() / 'results' / 'video_frames' / 'event_one'
frame_ext = '.png'  # PNG for lossless frames

# Biplots and cos_plots folders
biplots_folder = Path.cwd() / 'results' / 'biplots' / 'event_one'
cos_plots_folder = Path.cwd() / 'results' / 'cos_plots' / 'event_one'

# FPS settings
extraction_fps = 30.0           # extract frames at source FPS (or lower)
composite_fps = 30.0            # output video FPS (lower to slow playback)

# Composite output settings
max_width = 3840
max_height = 2160
gap_px = 10

# Optional slow-motion settings for composite (applies to all 3 panels)
slow_start = 23.0   # e.g., 2.0
slow_end = None     # e.g., 3.0
slow_factor = 24.0     # number of repeats within [slow_start, slow_end]

fps_label = str(int(composite_fps)) if float(composite_fps).is_integer() else str(composite_fps)
composite_filename_no_audio = f"video_biplots_cosplots_event_1_{fps_label}fps.mp4"
composite_filename_with_audio = f"video_biplots_cosplots_event_1_{fps_label}fps_audio.mp4"

print(f"Source video: {source_video}")
print(f"Frames dir: {video_frames_dir}")
print(f"Biplots: {biplots_folder}")
print(f"Cos plots: {cos_plots_folder}")
print(f"Composite (no audio): {Path.cwd() / 'results' / 'videos' / composite_filename_no_audio}")
print(f"Composite (with audio): {Path.cwd() / 'results' / 'videos' / composite_filename_with_audio}")


Source video: /Users/johnmadrid/Documents/PhD/EEDA_experiment copy/Events_Videos/event1-new.mp4
Frames dir: /Users/johnmadrid/GitHub/pca-driving-behavior/results/video_frames/event_one
Biplots: /Users/johnmadrid/GitHub/pca-driving-behavior/results/biplots/event_one
Cos plots: /Users/johnmadrid/GitHub/pca-driving-behavior/results/cos_plots/event_one
Composite (no audio): /Users/johnmadrid/GitHub/pca-driving-behavior/results/videos/video_biplots_cosplots_event_1_30fps.mp4
Composite (with audio): /Users/johnmadrid/GitHub/pca-driving-behavior/results/videos/video_biplots_cosplots_event_1_30fps_audio.mp4


In [31]:
# Probe source video properties
import cv2

cap = cv2.VideoCapture(str(source_video))
if not cap.isOpened():
    raise RuntimeError(f"Failed to open video: {source_video}")

fps_video = cap.get(cv2.CAP_PROP_FPS) or 0.0
frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH) or 0)
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT) or 0)
duration_s = (frame_count / fps_video) if fps_video else 0.0
cap.release()

print(f"Video FPS: {fps_video}")
print(f"Resolution: {width}x{height}")
print(f"Frames: {frame_count}")
print(f"Duration (s): {duration_s:.3f}")


Video FPS: 29.847700196433582
Resolution: 1920x1080
Frames: 273
Duration (s): 9.146


In [32]:
# Function: extract frames from video to PNG at target FPS
import cv2
from pathlib import Path


def extract_frames_from_video(source_path, out_dir, target_fps=None, ext='.png', downscale_max_width=None, downscale_max_height=None):
    source_path = Path(source_path)
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    cap = cv2.VideoCapture(str(source_path))
    if not cap.isOpened():
        raise RuntimeError(f"Failed to open video: {source_path}")

    fps_src = cap.get(cv2.CAP_PROP_FPS) or 30.0
    if target_fps is None or target_fps <= 0:
        target_fps = fps_src
    if target_fps > fps_src:
        target_fps = fps_src

    frame_interval = fps_src / target_fps
    frame_idx = 0
    next_save_idx = 0.0
    stem = source_path.stem

    saved = 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break

        if frame_idx + 1e-6 >= next_save_idx:
            h, w = frame.shape[:2]
            if downscale_max_width or downscale_max_height:
                sx = (downscale_max_width / w) if downscale_max_width else 1.0
                sy = (downscale_max_height / h) if downscale_max_height else 1.0
                scale = min(sx, sy, 1.0)
                if scale < 1.0:
                    new_w = int(round(w * scale))
                    new_h = int(round(h * scale))
                    frame = cv2.resize(frame, (new_w, new_h), interpolation=cv2.INTER_AREA)

            ts = frame_idx / fps_src
            filename = f"video_{stem}_{ts:.3f}{ext}"
            cv2.imwrite(str(out_dir / filename), frame)
            saved += 1
            next_save_idx += frame_interval

        frame_idx += 1

    cap.release()
    print(f"Saved {saved} frames to {out_dir}")


In [33]:
# Extract frames at extraction_fps to PNG
extract_frames_from_video(
    source_path=source_video,
    out_dir=video_frames_dir,
    target_fps=extraction_fps,
    ext=frame_ext,
)


Saved 258 frames to /Users/johnmadrid/GitHub/pca-driving-behavior/results/video_frames/event_one


In [42]:
# Function: build three-panel composite (video frame | biplot | cos_plot) without audio
import cv2
import numpy as np
import os
from pathlib import Path
import re
import bisect


def _extract_seconds_from_video_frame_filename(filename):
    m = re.search(r'_(\d+(?:\.\d+)?)\.png$', filename)
    if m:
        try:
            return float(m.group(1))
        except Exception:
            pass
    return float('inf')


def make_three_panel_composite_from_video_frames(
    video_frames_dir,
    biplots_dir,
    cosplots_dir,
    output_filename,
    fps,
    max_width=3840,
    max_height=2160,
    gap_px=10,
    allow_upscale=False,
    bg_color=(0, 0, 0),
):
    vdir = Path(video_frames_dir)
    bdir = Path(biplots_dir)
    cdir = Path(cosplots_dir)
    if not vdir.exists() or not vdir.is_dir():
        raise FileNotFoundError(f"Video frames dir not found: {vdir}")
    if not bdir.exists() or not bdir.is_dir():
        raise FileNotFoundError(f"Biplots dir not found: {bdir}")
    if not cdir.exists() or not cdir.is_dir():
        raise FileNotFoundError(f"Cos plots dir not found: {cdir}")

    vfiles = [f for f in os.listdir(vdir) if f.lower().endswith('.png')]
    if not vfiles:
        raise FileNotFoundError(f"No video frames found in {vdir}")
    vfiles.sort(key=_extract_seconds_from_video_frame_filename)

    bfiles = [f for f in os.listdir(bdir) if f.lower().endswith('.png')]
    cfiles = [f for f in os.listdir(cdir) if f.lower().endswith('.png')]
    if not bfiles:
        raise FileNotFoundError(f"No PNG images found in {bdir}")
    if not cfiles:
        raise FileNotFoundError(f"No PNG images found in {cdir}")

    # Build timestamp mappings for biplots and cos_plots
    b_ts, b_map = [], {}
    for f in bfiles:
        ts = _extract_timestamp_from_filename(f)
        if ts != float('inf'):
            b_ts.append(ts)
            b_map[ts] = f
    b_ts.sort()

    c_ts, c_map = [], {}
    for f in cfiles:
        ts = _extract_timestamp_from_filename(f)
        if ts != float('inf'):
            c_ts.append(ts)
            c_map[ts] = f
    c_ts.sort()

    if not b_ts or not c_ts:
        raise RuntimeError("Failed to parse timestamps from biplots/cos_plots filenames.")

    # Determine aspect ratios from first available files
    v0 = cv2.imread(str(vdir / vfiles[0]))
    b0 = cv2.imread(str(bdir / b_map[b_ts[0]]))
    c0 = cv2.imread(str(cdir / c_map[c_ts[0]]))
    if v0 is None or b0 is None or c0 is None:
        raise RuntimeError("Failed to read initial frames for sizing.")

    hv, wv = v0.shape[0], v0.shape[1]
    hb, wb = b0.shape[0], b0.shape[1]
    hc, wc = c0.shape[0], c0.shape[1]

    rv = wv / hv
    rb = wb / hb
    rc = wc / hc

    hcap_v = float('inf') if allow_upscale else hv
    hcap_b = float('inf') if allow_upscale else hb
    hcap_c = float('inf') if allow_upscale else hc

    height_limit_by_width = (max_width - 2 * gap_px) / (rv + rb + rc)
    target_h = int(round(min(hcap_v, hcap_b, hcap_c, max_height, height_limit_by_width)))
    if target_h <= 0:
        raise ValueError("Computed target height is invalid.")
    if target_h % 2 != 0:
        target_h -= 1

    def scaled_w(ratio, th):
        w = int(round(ratio * th))
        if w % 2 != 0:
            w -= 1
        return max(w, 2)

    wv_s = scaled_w(rv, target_h)
    wb_s = scaled_w(rb, target_h)
    wc_s = scaled_w(rc, target_h)

    while (wv_s + gap_px + wb_s + gap_px + wc_s) > max_width and target_h > 2:
        target_h -= 2
        wv_s = scaled_w(rv, target_h)
        wb_s = scaled_w(rb, target_h)
        wc_s = scaled_w(rc, target_h)

    out_w = wv_s + gap_px + wb_s + gap_px + wc_s
    if out_w % 2 != 0:
        if wc_s > 2:
            wc_s -= 2
        else:
            wb_s -= 2
        out_w = wv_s + gap_px + wb_s + gap_px + wc_s

    videos_dir = Path.cwd() / 'results' / 'videos'
    videos_dir.mkdir(parents=True, exist_ok=True)
    out_path = videos_dir / output_filename

    codec_preference = ['avc1', 'H264', 'X264', 'mp4v']
    writer = None
    for codec in codec_preference:
        tmp = cv2.VideoWriter(str(out_path), cv2.VideoWriter_fourcc(*codec), fps, (out_w, target_h))
        if tmp.isOpened():
            writer = tmp
            break
        else:
            tmp.release()
    if writer is None:
        raise RuntimeError("Failed to initialize VideoWriter for three-panel composite.")

    def nearest(ts_list, ts):
        i = bisect.bisect_left(ts_list, ts)
        if i == 0:
            return ts_list[0]
        if i == len(ts_list):
            return ts_list[-1]
        before = ts_list[i - 1]
        after = ts_list[i]
        return after if abs(after - ts) < abs(ts - before) else before

    try:
        for vf in vfiles:
            ts_v = _extract_seconds_from_video_frame_filename(vf)
            if ts_v == float('inf'):
                continue

            vi = cv2.imread(str(vdir / vf))
            if vi is None:
                continue

            tb = nearest(b_ts, ts_v)
            tc = nearest(c_ts, ts_v)

            bi = cv2.imread(str(bdir / b_map[tb]))
            ci = cv2.imread(str(cdir / c_map[tc]))
            if bi is None or ci is None:
                continue

            interp_v = cv2.INTER_AREA if (vi.shape[1] > wv_s or vi.shape[0] > target_h) else cv2.INTER_LANCZOS4
            interp_b = cv2.INTER_AREA if (bi.shape[1] > wb_s or bi.shape[0] > target_h) else cv2.INTER_LANCZOS4
            interp_c = cv2.INTER_AREA if (ci.shape[1] > wc_s or ci.shape[0] > target_h) else cv2.INTER_LANCZOS4

            vi_r = cv2.resize(vi, (wv_s, target_h), interpolation=interp_v)
            bi_r = cv2.resize(bi, (wb_s, target_h), interpolation=interp_b)
            ci_r = cv2.resize(ci, (wc_s, target_h), interpolation=interp_c)

            frame = np.full((target_h, out_w, 3), bg_color, dtype=np.uint8)
            x = 0
            frame[:, x : x + wv_s] = vi_r
            x += wv_s + gap_px
            frame[:, x : x + wb_s] = bi_r
            x += wb_s + gap_px
            frame[:, x : x + wc_s] = ci_r

            writer.write(frame)
    finally:
        writer.release()
        cv2.destroyAllWindows()

    return str(out_path)


In [48]:
# Override: three-panel composite using 50 fps index mapping with clamped hold-first/last
import os
import re
import bisect
from pathlib import Path
import cv2
import numpy as np


def make_three_panel_composite_from_video_frames(
    video_frames_dir,
    biplots_dir,
    cosplots_dir,
    output_filename,
    fps,
    max_width=3840,
    max_height=2160,
    gap_px=10,
    allow_upscale=False,
    bg_color=(0, 0, 0),
    images_fps=50.0,
    slow_start=None,
    slow_end=None,
    slow_factor=3,
    debug=True,
):
    vdir = Path(video_frames_dir)
    bdir = Path(biplots_dir)
    cdir = Path(cosplots_dir)
    if not vdir.exists() or not vdir.is_dir():
        raise FileNotFoundError(f"Video frames dir not found: {vdir}")
    if not bdir.exists() or not bdir.is_dir():
        raise FileNotFoundError(f"Biplots dir not found: {bdir}")
    if not cdir.exists() or not cdir.is_dir():
        raise FileNotFoundError(f"Cos plots dir not found: {cdir}")

    # Collect and sort video frames by embedded seconds
    vfiles = [f for f in os.listdir(vdir) if f.lower().endswith('.png')]
    if not vfiles:
        raise FileNotFoundError(f"No video frames found in {vdir}")
    def _vsec(name: str) -> float:
        m = re.search(r'_(\d+(?:\.\d+)?)\.png$', name)
        return float(m.group(1)) if m else float('inf')
    vfiles.sort(key=_vsec)

    # Collect and sort biplots/cos_plots by timestamp extracted with existing helper
    bfiles = [f for f in os.listdir(bdir) if f.lower().endswith('.png')]
    cfiles = [f for f in os.listdir(cdir) if f.lower().endswith('.png')]
    if not bfiles:
        raise FileNotFoundError(f"No PNG images found in {bdir}")
    if not cfiles:
        raise FileNotFoundError(f"No PNG images found in {cdir}")

    b_pairs = sorted(
        ((ts, f) for f in bfiles for ts in [_extract_timestamp_from_filename(f)] if ts != float('inf')),
        key=lambda x: x[0],
    )
    c_pairs = sorted(
        ((ts, f) for f in cfiles for ts in [_extract_timestamp_from_filename(f)] if ts != float('inf')),
        key=lambda x: x[0],
    )
    if not b_pairs or not c_pairs:
        raise RuntimeError("Failed to parse timestamps from biplots/cos_plots filenames.")

    b_ts = [ts for ts, _ in b_pairs]
    c_ts = [ts for ts, _ in c_pairs]
    event_start_s = min(b_ts[0], c_ts[0])

    # Sizing based on first frames
    v0 = cv2.imread(str(vdir / vfiles[0]))
    b0 = cv2.imread(str(bdir / b_pairs[0][1]))
    c0 = cv2.imread(str(cdir / c_pairs[0][1]))
    if v0 is None or b0 is None or c0 is None:
        raise RuntimeError("Failed to read initial frames for sizing.")

    hv, wv = v0.shape[0], v0.shape[1]
    hb, wb = b0.shape[0], b0.shape[1]
    hc, wc = c0.shape[0], c0.shape[1]

    rv = wv / hv
    rb = wb / hb
    rc = wc / hc

    hcap_v = float('inf') if allow_upscale else hv
    hcap_b = float('inf') if allow_upscale else hb
    hcap_c = float('inf') if allow_upscale else hc

    height_limit_by_width = (max_width - 2 * gap_px) / (rv + rb + rc)
    target_h = int(round(min(hcap_v, hcap_b, hcap_c, max_height, height_limit_by_width)))
    if target_h <= 0:
        raise ValueError("Computed target height is invalid.")
    if target_h % 2 != 0:
        target_h -= 1

    def scaled_w(ratio, th):
        w = int(round(ratio * th))
        if w % 2 != 0:
            w -= 1
        return max(w, 2)

    wv_s = scaled_w(rv, target_h)
    wb_s = scaled_w(rb, target_h)
    wc_s = scaled_w(rc, target_h)

    while (wv_s + gap_px + wb_s + gap_px + wc_s) > max_width and target_h > 2:
        target_h -= 2
        wv_s = scaled_w(rv, target_h)
        wb_s = scaled_w(rb, target_h)
        wc_s = scaled_w(rc, target_h)

    out_w = wv_s + gap_px + wb_s + gap_px + wc_s
    if out_w % 2 != 0:
        if wc_s > 2:
            wc_s -= 2
        else:
            wb_s -= 2
        out_w = wv_s + gap_px + wb_s + gap_px + wc_s

    videos_dir = Path.cwd() / 'results' / 'videos'
    videos_dir.mkdir(parents=True, exist_ok=True)
    out_path = videos_dir / output_filename

    codec_preference = ['avc1', 'H264', 'X264', 'mp4v']
    writer = None
    for codec in codec_preference:
        tmp = cv2.VideoWriter(str(out_path), cv2.VideoWriter_fourcc(*codec), fps, (out_w, target_h))
        if tmp.isOpened():
            writer = tmp
            break
        else:
            tmp.release()
    if writer is None:
        raise RuntimeError("Failed to initialize VideoWriter for three-panel composite.")

    if debug:
        v_first_t = _vsec(vfiles[0])
        v_last_t = _vsec(vfiles[-1])
        print(
            f"Aligning 30fps video frames ({len(vfiles)}) to images at {images_fps}fps: "
            f"event_start={event_start_s:.3f}s, video_t=[{v_first_t:.3f},{v_last_t:.3f}], "
            f"biplots={len(b_pairs)}, cos_plots={len(c_pairs)}"
        )

    try:
        for vf in vfiles:
            t_v = _vsec(vf)
            # Map video time to image index at images_fps, clamp to [0, N-1]
            idx = int(round(t_v * images_fps))
            idx_b = min(max(idx, 0), len(b_pairs) - 1)
            idx_c = min(max(idx, 0), len(c_pairs) - 1)

            vi = cv2.imread(str(vdir / vf))
            bi = cv2.imread(str(bdir / b_pairs[idx_b][1]))
            ci = cv2.imread(str(cdir / c_pairs[idx_c][1]))
            if vi is None or bi is None or ci is None:
                continue

            interp_v = cv2.INTER_AREA if (vi.shape[1] > wv_s or vi.shape[0] > target_h) else cv2.INTER_LANCZOS4
            interp_b = cv2.INTER_AREA if (bi.shape[1] > wb_s or bi.shape[0] > target_h) else cv2.INTER_LANCZOS4
            interp_c = cv2.INTER_AREA if (ci.shape[1] > wc_s or ci.shape[0] > target_h) else cv2.INTER_LANCZOS4

            vi_r = cv2.resize(vi, (wv_s, target_h), interpolation=interp_v)
            bi_r = cv2.resize(bi, (wb_s, target_h), interpolation=interp_b)
            ci_r = cv2.resize(ci, (wc_s, target_h), interpolation=interp_c)

            frame = np.full((target_h, out_w, 3), bg_color, dtype=np.uint8)
            x = 0
            frame[:, x : x + wv_s] = vi_r
            x += wv_s + gap_px
            frame[:, x : x + wb_s] = bi_r
            x += wb_s + gap_px
            frame[:, x : x + wc_s] = ci_r

            # Slow motion: repeat frames when ORIGINAL image timestamps fall within [slow_start, slow_end]
            repeats = 1
            if slow_start is not None and slow_end is not None:
                ts_b = b_pairs[idx_b][0]
                ts_c = c_pairs[idx_c][0]
                if (slow_start <= ts_b <= slow_end) or (slow_start <= ts_c <= slow_end):
                    repeats = max(int(slow_factor), 1)
            for _ in range(repeats):
                writer.write(frame)
    finally:
        writer.release()
        cv2.destroyAllWindows()

    return str(out_path)


In [49]:
# Build three-panel composite (no audio)
output_no_audio = make_three_panel_composite_from_video_frames(
    video_frames_dir=video_frames_dir,
    biplots_dir=biplots_folder,
    cosplots_dir=cos_plots_folder,
    output_filename=composite_filename_no_audio,
    fps=composite_fps,
    max_width=max_width,
    max_height=max_height,
    gap_px=gap_px,
    slow_start=23.0,
    slow_end=24.0,
    slow_factor=slow_factor,
)
print(f"Saved composite (no audio) to: {output_no_audio}")


Aligning 30fps video frames (258) to images at 50.0fps: event_start=20.840s, video_t=[0.000,8.610], biplots=508, cos_plots=508
Saved composite (no audio) to: /Users/johnmadrid/GitHub/pca-driving-behavior/results/videos/video_biplots_cosplots_event_1_30fps.mp4


In [53]:
# slow_start = 23.0
# slow_end = 24.0
# slow_factor = 3

In [ ]:
# Mux audio from the source video into the composite using ffmpeg, applying global + windowed slow audio
import shutil
import subprocess
from pathlib import Path
import os
slow_start = 23.0
slow_end = 24.0
slow_factor = 3

videos_dir = Path.cwd() / 'results' / 'videos'
composite_in = videos_dir / composite_filename_no_audio
composite_out = videos_dir / composite_filename_with_audio

# Compute event start from image timestamps to align absolute times to clip timeline
def _scan_event_start(bdir: Path, cdir: Path):
    b_ts = []
    c_ts = []
    for f in os.listdir(bdir):
        if f.lower().endswith('.png'):
            ts = _extract_timestamp_from_filename(f)
            if ts != float('inf'):
                b_ts.append(ts)
    for f in os.listdir(cdir):
        if f.lower().endswith('.png'):
            ts = _extract_timestamp_from_filename(f)
            if ts != float('inf'):
                c_ts.append(ts)
    if not b_ts or not c_ts:
        return None
    return min(min(b_ts), min(c_ts))

# Build atempo chain within [0.5, 2.0]
def _build_atempo_chain(rate: float) -> str:
    if rate <= 0:
        rate = 1.0
    r = rate
    parts = []
    while r < 0.5:
        parts.append(0.5)
        r /= 0.5
    while r > 2.0:
        parts.append(2.0)
        r /= 2.0
    parts.append(r)
    return ",".join(f"atempo={p:.6f}" for p in parts)

# Probe stream start times to compensate AV offset
def _probe_stream_start_time(ffprobe_bin: str, path: Path, selector: str):
    try:
        r = subprocess.run(
            [ffprobe_bin, '-v', 'error', '-select_streams', selector,
             '-show_entries', 'stream=start_time',
             '-of', 'default=noprint_wrappers=1:nokey=1', str(path)],
            capture_output=True, text=True, check=False
        )
        out = (r.stdout or '').strip()
        if out and out != 'N/A':
            return float(out)
    except Exception:
        pass
    return None

ffmpeg = shutil.which('ffmpeg')
ffprobe = shutil.which('ffprobe')
if ffmpeg is None or ffprobe is None:
    print("ffmpeg/ffprobe not found. Install via Homebrew: brew install ffmpeg.")
else:
    use_slow = slow_start is not None and slow_end is not None and slow_factor and slow_factor > 1
    event_start = _scan_event_start(biplots_folder, cos_plots_folder) if use_slow else None

    # Global slowdown to match composite_fps vs extraction_fps
    try:
        s_global = float(composite_fps) / float(extraction_fps)
    except Exception:
        s_global = 1.0

    if use_slow and event_start is not None:
        # Convert absolute slow window to clip-relative times (original timeline)
        clip_start = max(0.0, float(slow_start) - float(event_start))
        clip_end = max(clip_start, float(slow_end) - float(event_start))

        # Compensate any AV offset in the source container
        a_start = _probe_stream_start_time(ffprobe, source_video, 'a:0')
        v_start = _probe_stream_start_time(ffprobe, source_video, 'v:0')
        av_offset = 0.0
        if a_start is not None and v_start is not None:
            av_offset = a_start - v_start
        clip_start_adj = max(0.0, clip_start + av_offset)
        clip_end_adj = max(clip_start_adj + 1e-3, clip_end + av_offset)

        # Chains: a0/a2 at s_global, a1 at s_global/slow_factor
        a0_chain = _build_atempo_chain(s_global)
        a1_chain = _build_atempo_chain(max(s_global / float(slow_factor), 1e-6))
        a2_chain = a0_chain

        filter_complex = (
            f"[1:a]asplit=3[a0][a1][a2];"
            f"[a0]atrim=0:{clip_start_adj:.6f},asetpts=PTS-STARTPTS,{a0_chain}[a0t];"
            f"[a1]atrim={clip_start_adj:.6f}:{clip_end_adj:.6f},asetpts=PTS-STARTPTS,{a1_chain}[a1t];"
            f"[a2]atrim=start={clip_end_adj:.6f},asetpts=PTS-STARTPTS,{a2_chain}[a2t];"
            f"[a0t][a1t][a2t]concat=n=3:v=0:a=1[aout]"
        )
        cmd = [
            ffmpeg, '-y',
            '-i', str(composite_in),
            '-i', str(source_video),
            '-filter_complex', filter_complex,
            '-map', '0:v:0',
            '-map', '[aout]',
            '-c:v', 'copy',
            '-c:a', 'aac',
            '-shortest',
            str(composite_out),
        ]
        print(
            f"event_start={event_start:.6f}, clip_slow=[{clip_start:.6f},{clip_end:.6f}], "
            f"av_offset={av_offset:.6f} -> adj=[{clip_start_adj:.6f},{clip_end_adj:.6f}], "
            f"slow_factor={slow_factor}, s_global={s_global:.6f}"
        )
        print('Running:', ' '.join(cmd))
        res = subprocess.run(cmd, capture_output=True, text=True)
        if res.returncode != 0:
            print('ffmpeg error:', res.stderr or res.stdout)
        else:
            print(f'Saved composite with audio to: {composite_out}')
    else:
        # No windowed slow: apply only global slowdown to entire audio
        a_chain = _build_atempo_chain(s_global)
        filter_complex = f"[1:a]{a_chain}[aout]"
        cmd = [
            ffmpeg, '-y',
            '-i', str(composite_in),
            '-i', str(source_video),
            '-filter_complex', filter_complex,
            '-map', '0:v:0',
            '-map', '[aout]',
            '-c:v', 'copy',
            '-c:a', 'aac',
            '-shortest',
            str(composite_out),
        ]
        print(f"s_global={s_global:.6f}")
        print('Running:', ' '.join(cmd))
        res = subprocess.run(cmd, capture_output=True, text=True)
        if res.returncode != 0:
            print('ffmpeg error:', res.stderr or res.stdout)
        else:
            print(f'Saved composite with audio to: {composite_out}')


event_start=20.840000, clip_slow=[2.160000,3.160000], slow_factor=3, s_global=1.000000
Running: /opt/homebrew/bin/ffmpeg -y -i /Users/johnmadrid/GitHub/pca-driving-behavior/results/videos/video_biplots_cosplots_event_1_30fps.mp4 -i /Users/johnmadrid/Documents/PhD/EEDA_experiment copy/Events_Videos/event1-new.mp4 -filter_complex [1:a]asplit=3[a0][a1][a2];[a0]atrim=0:2.160000,asetpts=PTS-STARTPTS,atempo=1.000000[a0t];[a1]atrim=2.160000:3.160000,asetpts=PTS-STARTPTS,atempo=0.500000,atempo=0.666667[a1t];[a2]atrim=start=3.160000,asetpts=PTS-STARTPTS,atempo=1.000000[a2t];[a0t][a1t][a2t]concat=n=3:v=0:a=1[aout] -map 0:v:0 -map [aout] -c:v copy -c:a aac -shortest /Users/johnmadrid/GitHub/pca-driving-behavior/results/videos/video_biplots_cosplots_event_1_30fps_audio.mp4
Saved composite with audio to: /Users/johnmadrid/GitHub/pca-driving-behavior/results/videos/video_biplots_cosplots_event_1_30fps_audio.mp4
